# 🔥 KURE Fine-tuning v2: Stage 2 전용 파인튜닝

## 📋 v1 대비 변경점 (v1 실패 원인 분석 후 재설계)

| 항목 | v1 (실패) | v2 (수정) |
|------|---------|--------|
| Stage 1 파인튜닝 | ✅ 시도 (→ 실패) | ❌ 제거 (v3 frozen 유지, F1=0.9877) |
| backbone 공유 | Stage 1/2 동일 객체 공유 | ✅ Stage 2 전용 독립 backbone |
| dropout | 0.5 (과함) | ✅ 0.2 (BERT 파인튜닝 권장값) |
| Stage 2 classifier 초기화 | random | ✅ v3 학습 가중치 warm-start |
| Stage 2 lr_head | 3e-4 | ✅ 1e-4 (warm-start 시 작게) |
| Stage 1 Focal Loss | ❌ 미사용 (불균형 무시) | — (Stage 1 파인튜닝 없음) |

## ❌ v1 실패 원인 요약
1. **Stage 1**: 86:14 클래스 불균형 + Focal Loss 미사용 + dropout 0.5 → 다수 클래스(부정)로 수렴  
2. **Stage 2**: Stage 1이 손상시킨 backbone을 그대로 이어받아 연쇄 실패

## 🎯 목표
- Stage 1: v3 frozen 모델 그대로 유지 (F1 0.9877, 개선 불필요)
- Stage 2: F1 0.4811 → **0.55+ 목표** (warm-start + backbone 파인튜닝)

---

## 1. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

# 필수 라이브러리
!pip install -q sentence-transformers transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 상수 및 경로 정의

In [ ]:
STAGE2_CATEGORIES = {
    0: "정서적_고갈",
    1: "좌절_압박",
    2: "부정적_대인관계",
    3: "자기비하"
}

# CSV 경로 (FineTune v1과 동일)
DATA_PATH = "/content/drive/MyDrive/Burnout/dataset"

# v3 모델 경로 (v3 노트북에서 저장한 위치)
V3_MODEL_PATH = "/content/drive/MyDrive/Burnout"

print("✅ 상수 정의 완료")
print(f"   Stage 2: {list(STAGE2_CATEGORIES.values())}")
print(f"   DATA_PATH:      {DATA_PATH}")
print(f"   V3_MODEL_PATH:  {V3_MODEL_PATH}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print(f"\n📂 파일 존재 여부 확인")

# CSV 파일 확인 (Stage 2만 필요)
csv_files = ['stage2_train_v3.csv', 'stage2_val_v3.csv']
for f in csv_files:
    path = f"{DATA_PATH}/{f}"
    exists = os.path.exists(path)
    print(f"   {'✅' if exists else '❌'} {f}")

# v3 Stage 2 모델 확인 (warm-start용)
v3_s2_path = f"{V3_MODEL_PATH}/stage2_model_v3.pt"
print(f"\n   {'✅' if os.path.exists(v3_s2_path) else '⚠️ 없음 (random init 사용)'} stage2_model_v3.pt (warm-start용)")

## 4. Stage 2 데이터 로드 및 증강

Stage 1은 파인튜닝하지 않으므로 Stage 2 데이터만 로드합니다.

In [ ]:
s2_train = pd.read_csv(f"{DATA_PATH}/stage2_train_v3.csv")
s2_val   = pd.read_csv(f"{DATA_PATH}/stage2_val_v3.csv")

print("✅ Stage 2 데이터 로드 완료")
print(f"   Train: {len(s2_train):,}개  |  Val: {len(s2_val):,}개")
print(f"\n   Train 분포:")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train['label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

In [ ]:
def random_mix_augmentation(df, label_col='label', augment_ratio=0.3,
                             min_merge=1, max_merge=3, seed=42):
    rng = np.random.default_rng(seed)
    augmented_rows = []
    for label in sorted(df[label_col].unique()):
        texts = df[df[label_col] == label]['text'].tolist()
        n_generate = max(1, int(len(texts) * augment_ratio))
        for _ in range(n_generate):
            k = rng.integers(min_merge, max_merge + 1)
            k = min(k, len(texts))
            indices = rng.choice(len(texts), size=k, replace=False)
            combined = ' '.join([texts[i] for i in indices])
            augmented_rows.append({'text': combined, label_col: label})
    augmented_df = pd.DataFrame(augmented_rows)
    result = pd.concat([df, augmented_df], ignore_index=True).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    return result


s2_train_aug = random_mix_augmentation(s2_train, augment_ratio=0.3)

print("✅ 증강 완료")
print(f"   S2 Train: {len(s2_train):,} → {len(s2_train_aug):,}")
print(f"   증강 후 분포:")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train_aug['label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

## 5. KURE 모델 로드 및 Fine-tuning 설정

> **핵심 변경**: Stage 2 전용 독립 backbone 사용  
> Stage 1과 backbone을 공유하지 않으므로 Stage 1 학습 결과에 영향받지 않음

In [ ]:
print("🔄 KURE 모델 로딩 중...")
st_model = SentenceTransformer('nlpai-lab/KURE-v1')

# Stage 2 전용 backbone 추출 (독립 객체)
transformer_module = st_model[0]
tokenizer          = transformer_module.tokenizer
backbone           = transformer_module.auto_model  # Stage 2 전용

HIDDEN_SIZE = backbone.config.hidden_size
NUM_LAYERS  = len(backbone.encoder.layer)

print(f"✅ KURE 로드 완료")
print(f"   Hidden Size: {HIDDEN_SIZE}")
print(f"   Encoder 레이어 수: {NUM_LAYERS}")
print(f"   → Stage 2 전용 backbone으로 사용 (Stage 1과 독립)")

## 6. 모델 아키텍처 정의

**v1 대비 변경**:
- `dropout 0.5 → 0.2` (BERT 파인튜닝 권장값)
- Classifier head 구조는 v3 `BurnoutClassifier`와 동일하게 유지 → warm-start 가중치 전이 가능

In [ ]:
class FineTunedBurnoutClassifier(nn.Module):
    """
    KURE backbone + 분류기 헤드
    - backbone 상위 N개 레이어만 해제, 나머지 동결
    - classifier head 구조가 v3 BurnoutClassifier와 동일 → warm-start 전이 가능
    """
    def __init__(self, backbone, num_classes, num_unfreeze_layers=3, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        num_total = len(backbone.encoder.layer)

        # 전체 동결
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 상위 N개 레이어 해제
        for i in range(num_total - num_unfreeze_layers, num_total):
            for param in self.backbone.encoder.layer[i].parameters():
                param.requires_grad = True

        # 상단 LayerNorm 해제
        if hasattr(self.backbone, 'LayerNorm'):
            for param in self.backbone.LayerNorm.parameters():
                param.requires_grad = True

        hidden_size = backbone.config.hidden_size  # 1024
        # ⚠️ v3 BurnoutClassifier와 동일한 구조 (warm-start 호환)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),      # ← 0.2 (v1의 0.5에서 감소)
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"   학습 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * mask_expanded, 1) / \
               torch.clamp(mask_expanded.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs    = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return self.classifier(embeddings)


class TextDataset(Dataset):
    """텍스트 → 토크나이징 → Dataset"""
    def __init__(self, texts, labels, tokenizer, max_length=256):
        print(f"   토크나이징 중... ({len(texts):,}개)", end=" ")
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding='max_length',   # 전체 동일 길이 → DataLoader 효율 향상
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        print("완료")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }


print("✅ 모델/데이터셋 클래스 정의 완료")
print(f"   dropout: 0.2 (v1의 0.5에서 감소)")

## 7. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        smooth_targets = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
        smooth_targets = smooth_targets * (1 - self.label_smoothing) + \
                         self.label_smoothing / num_classes
        log_probs = F.log_softmax(inputs, dim=-1)
        probs = torch.exp(log_probs)
        focal_weight = (1 - probs) ** self.gamma
        loss = -focal_weight * smooth_targets * log_probs
        if self.alpha is not None:
            loss = loss * self.alpha[targets].unsqueeze(1)
        return loss.sum(dim=-1).mean()


def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


print("✅ Loss Functions 정의 완료")

## 8. Fine-tuning 학습 함수 (Stage 2 전용)

In [ ]:
def finetune_stage2(
    model, train_dataset, val_dataset, train_labels,
    epochs=20, batch_size=16, lr_backbone=1e-5, lr_head=1e-4,
    weight_decay=1e-4, patience=7, min_delta=0.001,
    label_smoothing=0.15, warmup_epochs=3, device='cuda'
):
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    num_classes   = len(np.unique(train_labels))
    class_weights = compute_class_weights(train_labels, num_classes).to(device)

    # Stage 2는 항상 Focal Loss 사용 (v1 Stage 1의 실수 반복 안 함)
    criterion = FocalLoss(gamma=2.0, alpha=class_weights,
                          label_smoothing=label_smoothing)

    # 차등 learning rate
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params     = list(model.classifier.parameters())
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': lr_head}
    ], weight_decay=weight_decay)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [],
               'val_acc': [], 'val_f1': [], 'lr_backbone': [], 'lr_head': []}
    best_val_f1, best_state, patience_counter = 0, None, 0

    print(f"\n{'='*65}")
    print(f"🚀 Stage 2 Fine-tuning 시작")
    print(f"   epochs={epochs}, batch={batch_size}")
    print(f"   KURE 레이어 lr={lr_backbone}  |  Head lr={lr_head}")
    print(f"   Loss: FocalLoss (γ=2.0, smoothing={label_smoothing})")
    print(f"   Class Weights: {class_weights.cpu().numpy().round(3)}")
    print(f"{'='*65}")

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for batch in pbar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, pred = torch.max(logits, 1)
            train_total   += labels.size(0)
            train_correct += (pred == labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()
        train_acc      = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        all_preds, all_labels_list = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['labels'].to(device)

                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, labels)
                val_loss += loss.item()
                _, pred = torch.max(logits, 1)
                val_total   += labels.size(0)
                val_correct += (pred == labels).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels_list.extend(labels.cpu().numpy())

        val_acc      = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        val_f1       = f1_score(all_labels_list, all_preds, average='weighted')
        lrs          = [g['lr'] for g in optimizer.param_groups]

        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr_backbone'].append(lrs[0])
        history['lr_head'].append(lrs[1])

        print(f"Epoch {epoch+1:2d} | Loss: {avg_train_loss:.4f} | "
              f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}% | "
              f"F1: {val_f1:.4f} | lr_bb: {lrs[0]:.2e}")

        if val_f1 > best_val_f1 + min_delta:
            best_val_f1 = val_f1
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"         ✅ Best F1: {best_val_f1:.4f} (saved)")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⚠️ Early Stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"\n✅ Best Model 복원 (F1: {best_val_f1:.4f})")

    return model, history, {'best_f1': best_val_f1, 'best_acc': max(history['val_acc'])}


print("✅ Fine-tuning 함수 정의 완료")

## 9. Stage 2 데이터 토크나이징

In [ ]:
MAX_LENGTH = 256

print("🔄 Stage 2 토크나이징...")
s2_train_ds = TextDataset(s2_train_aug['text'], s2_train_aug['label'].values, tokenizer, MAX_LENGTH)
s2_val_ds   = TextDataset(s2_val['text'],       s2_val['label'].values,       tokenizer, MAX_LENGTH)

print(f"\n✅ 토크나이징 완료")
print(f"   S2 Train: {len(s2_train_ds):,}  |  Val: {len(s2_val_ds):,}")

## 10. Stage 2 모델 생성 + v3 가중치 Warm-start

### Warm-start 원리
```
v3 BurnoutClassifier (frozen KURE + 학습된 classifier)
         ↓ classifier.* 키만 추출
FineTunedBurnoutClassifier (KURE backbone + 동일 classifier 구조)
         → backbone: KURE pretrained (frozen → 상위 3레이어 해제)
         → classifier: v3 학습 가중치 로드 ✅
```

v3 모델 파일이 없으면 random initialization으로 진행합니다.

In [ ]:
print("="*65)
print("📊 STAGE 2: 4개 번아웃 카테고리 Fine-tuning")
print("="*65)

stage2_ft = FineTunedBurnoutClassifier(
    backbone=backbone,
    num_classes=4,
    num_unfreeze_layers=3,
    dropout=0.2           # v1의 0.5에서 감소
).to(device)

# ── v3 Stage 2 classifier 가중치 warm-start ──
v3_s2_path = f"{V3_MODEL_PATH}/stage2_model_v3.pt"

if os.path.exists(v3_s2_path):
    print("\n🔄 v3 Stage 2 가중치 로딩 (warm-start)...")
    # weights_only=False: PyTorch 2.6+ 에서 numpy scalar 포함 체크포인트 로드 시 필요
    checkpoint = torch.load(v3_s2_path, map_location=device, weights_only=False)
    v3_state   = checkpoint['model_state_dict']

    # classifier.* 키만 필터링 (backbone 키는 없음 — v3는 frozen KURE)
    classifier_keys = {k: v for k, v in v3_state.items() if k.startswith('classifier.')}

    # strict=False: backbone은 KURE pretrained 유지, classifier만 교체
    missing, unexpected = stage2_ft.load_state_dict(
        {**stage2_ft.state_dict(), **{k: v.to(device) for k, v in classifier_keys.items()}},
        strict=True
    )

    print(f"✅ Warm-start 완료")
    print(f"   전이된 classifier 키: {len(classifier_keys)}개")
    for k in classifier_keys.keys():
        print(f"     {k}: {classifier_keys[k].shape}")
else:
    print("\n⚠️ v3 모델 없음 → Random initialization으로 진행")
    print(f"   (경로 확인: {v3_s2_path})")

## 11. Stage 2 Fine-tuning 실행

In [ ]:
S2_FT_CONFIG = {
    'epochs':          20,
    'batch_size':      16,
    'lr_backbone':     1e-5,   # backbone: 매우 보수적
    'lr_head':         1e-4,   # head: warm-start라 3e-4 → 1e-4로 감소
    'weight_decay':    1e-4,
    'patience':        7,
    'label_smoothing': 0.15,
    'warmup_epochs':   3,
}

stage2_ft, s2_history, s2_best = finetune_stage2(
    model=stage2_ft,
    train_dataset=s2_train_ds,
    val_dataset=s2_val_ds,
    train_labels=s2_train_aug['label'].values,
    **S2_FT_CONFIG,
    device=device
)

print(f"\n📈 Stage 2 최종 결과:")
print(f"   Best Val Acc: {s2_best['best_acc']:.2f}%")
print(f"   Best Val F1:  {s2_best['best_f1']:.4f}")
print(f"   v3 대비 F1 변화: {s2_best['best_f1'] - 0.4811:+.4f}")

## 12. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(s2_history['train_loss'], label='Train')
axes[0].plot(s2_history['val_loss'],   label='Val')
axes[0].set_title('Stage 2 Fine-tuned: Loss')
axes[0].legend()

axes[1].plot(s2_history['train_acc'], label='Train')
axes[1].plot(s2_history['val_acc'],   label='Val')
axes[1].axhline(y=48.1, color='r', linestyle='--', label='v3 기준선 (48.1%)')
axes[1].set_title(f'Stage 2: Accuracy (Best: {s2_best["best_acc"]:.1f}%)')
axes[1].legend()

axes[2].plot(s2_history['val_f1'])
axes[2].axhline(y=0.4811, color='r', linestyle='--', label='v3 기준선 (0.4811)')
axes[2].set_title(f'Stage 2: Val F1 (Best: {s2_best["best_f1"]:.4f})')
axes[2].legend()

plt.suptitle('KURE Fine-tuning v2 — Stage 2 Only', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/training_curves_v4.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ 그래프 저장 완료")

## 13. 상세 평가

In [ ]:
def evaluate_ft_model(model, dataset, categories, stage_name, device='cuda'):
    model.eval()
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids, attention_mask)
            _, pred = torch.max(logits, 1)
            all_preds.extend(pred.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    print(f"\n{'='*60}")
    print(f"📊 {stage_name} 상세 평가")
    print(f"{'='*60}")
    print(classification_report(
        all_labels_list, all_preds,
        target_names=list(categories.values()), digits=4
    ))
    cm = confusion_matrix(all_labels_list, all_preds)
    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=[f"실제_{v}" for v in categories.values()],
        columns=[f"예측_{v}" for v in categories.values()]
    ))
    return all_preds, all_labels_list


evaluate_ft_model(stage2_ft, s2_val_ds, STAGE2_CATEGORIES, "Stage 2 Fine-tuned (v4)", device)

## 14. 모델 저장

- 저장명: `stage2_model_v4.pt` (기존 v3 모델 보존)
- Stage 1은 기존 `stage1_model_v3.pt` 그대로 사용

In [ ]:
improvement = s2_best['best_f1'] - 0.4811

if improvement > 0:
    save_path = f"{DATA_PATH}/stage2_model_v4.pt"
    torch.save({
        'model_state_dict':    stage2_ft.state_dict(),
        'backbone_model':      'nlpai-lab/KURE-v1',
        'num_unfreeze_layers': 3,
        'num_classes':         4,
        'dropout':             0.2,
        'categories':          STAGE2_CATEGORIES,
        'config':              S2_FT_CONFIG,
        'best_metrics':        s2_best,
        'data_version':        'v4',
        'warm_start_from':     'stage2_model_v3.pt',
    }, save_path)
    print(f"✅ Stage 2 저장: {save_path}")
    print(f"   v3 대비 F1 향상: {improvement:+.4f} ({'✅ 개선됨' if improvement > 0.02 else '↔ 미미한 향상'})")
    print(f"\n📋 서버 배포 시 변경사항:")
    print(f"   analyzer.py: stage2_model.pt → stage2_model_v4.pt 로 교체 필요")
    print(f"   (Stage 1은 stage1_model_v3.pt 그대로 유지)")
else:
    print(f"❌ 개선 없음 (v3 F1: 0.4811 / v4 F1: {s2_best['best_f1']:.4f})")
    print(f"   stage2_model_v4.pt 저장 생략 — v3 모델 유지 권장")
    print(f"\n💡 추가 시도 가능한 방법:")
    print(f"   1. num_unfreeze_layers=4 또는 5로 증가")
    print(f"   2. lr_backbone=2e-5 로 상향")
    print(f"   3. epochs=30으로 증가 (patience도 10으로)")

## 15. 추론 테스트

In [ ]:
# Stage 1은 v3 frozen 모델 로드
# Stage 2는 방금 학습한 fine-tuned 모델 사용
from sentence_transformers import SentenceTransformer as ST
import torch.nn as nn

# v3 Stage 1 모델 로드 (frozen KURE 기반)
class BurnoutClassifier(nn.Module):
    """v3 frozen KURE 모델 추론용"""
    def __init__(self, input_dim=1024, hidden_dim=256, num_classes=2, dropout=0.2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)


STAGE1_CATEGORIES = {0: "긍정", 1: "부정"}
v3_s1_path = f"{V3_MODEL_PATH}/stage1_model_v3.pt"

if os.path.exists(v3_s1_path):
    # weights_only=False: PyTorch 2.6+ 호환
    ckpt_s1 = torch.load(v3_s1_path, map_location=device, weights_only=False)
    stage1_v3 = BurnoutClassifier(
        input_dim=ckpt_s1.get('embedding_dim', 1024),
        hidden_dim=ckpt_s1.get('hidden_dim', 256),
        num_classes=2,
        dropout=ckpt_s1.get('dropout', 0.5)
    ).to(device)
    stage1_v3.load_state_dict(ckpt_s1['model_state_dict'])
    stage1_v3.eval()
    print("✅ Stage 1 v3 모델 로드 완료")
    kure_infer = ST('nlpai-lab/KURE-v1').to(device)
else:
    print(f"⚠️ Stage 1 v3 모델 없음 ({v3_s1_path})")
    stage1_v3 = None


def predict_2stage_v4(text, kure, stage1, stage2_ft, max_length=256, device='cuda'):
    """Stage 1: v3 frozen / Stage 2: v4 fine-tuned"""
    stage2_ft.eval()

    encoding = tokenizer(
        text, truncation=True, padding='max_length',
        max_length=max_length, return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        # Stage 1: KURE embedding + frozen classifier
        emb = kure.encode(text, convert_to_tensor=True).unsqueeze(0).to(device)
        s1_logits = stage1(emb)
        s1_probs  = F.softmax(s1_logits, dim=-1)[0]
        s1_pred   = torch.argmax(s1_logits, dim=-1).item()

        result = {
            'text': text,
            'stage1': {
                'category': STAGE1_CATEGORIES[s1_pred],
                'confidence': s1_probs[s1_pred].item(),
                'probs': {STAGE1_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s1_probs)}
            },
            'stage2': None
        }

        if s1_pred == 1:  # 부정
            # Stage 2: fine-tuned backbone
            s2_logits = stage2_ft(input_ids, attention_mask)
            s2_probs  = F.softmax(s2_logits, dim=-1)[0]
            s2_pred   = torch.argmax(s2_logits, dim=-1).item()
            result['stage2'] = {
                'category': STAGE2_CATEGORIES[s2_pred],
                'confidence': s2_probs[s2_pred].item(),
                'probs': {STAGE2_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s2_probs)}
            }
    return result


if stage1_v3 is not None:
    test_texts = [
        "오늘도 야근이었다. 집에 오니 아무것도 하기 싫고 그냥 쓰러지고 싶었다.",
        "팀장이 또 내 앞에서 나를 무시했다. 너무 억울하고 화가 난다.",
        "요즘 들어 출근이 너무 싫다. 사람들 얼굴 보기도 싫고 그냥 다 피하고 싶다.",
        "나는 왜 이것밖에 못 할까. 이러니 아무도 날 인정 안 하지.",
        "오늘 발표가 잘 됐다! 팀장님도 칭찬해 주셔서 기분이 좋았다.",
        "잠을 못 잤더니 온종일 멍했다. 아무것도 집중이 안 되고 너무 지쳤다."
    ]

    print("🧪 일기 스타일 텍스트 테스트 (Stage 1: v3 frozen / Stage 2: v4 fine-tuned)")
    for text in test_texts:
        r  = predict_2stage_v4(text, kure_infer, stage1_v3, stage2_ft, device=device)
        s1 = r['stage1']
        s2 = r['stage2']
        print(f"\n📝 {r['text'][:50]}...")
        print(f"   Stage 1: {s1['category']} ({s1['confidence']:.1%})")
        if s2:
            print(f"   Stage 2: {s2['category']} ({s2['confidence']:.1%})")

## 16. 최종 요약

In [ ]:
print("="*70)
print("📋 Fine-tuning v2 결과 요약")
print("="*70)

improvement = s2_best['best_f1'] - 0.4811

print("\n🎯 성능 비교")
print("-"*70)
print(f"  {'':30s}  {'v3 (Frozen)':>14s}  {'v4 (Fine-tuned)':>16s}")
print(f"  {'Stage 1 F1':30s}  {'0.9877':>14s}  {'유지 (파인튜닝 없음)':>16s}")
print(f"  {'Stage 2 F1':30s}  {'0.4811':>14s}  {s2_best['best_f1']:>16.4f}")
print(f"  {'Stage 2 Acc':30s}  {'48.1%':>14s}  {s2_best['best_acc']:>15.2f}%")
print(f"\n  Stage 2 F1 변화: {'+' if improvement >= 0 else ''}{improvement:.4f}")

if improvement > 0.02:
    print("  ✅ Fine-tuning 효과 있음 → v4 모델 사용 권장")
elif improvement > 0:
    print("  ↔ 미미한 향상 → 추가 실험 고려")
else:
    print("  ❌ 개선 없음 → v3 모델 유지 / 하이퍼파라미터 재조정 필요")

print("\n🔧 v2 설계 변경 요약")
print("-"*70)
print("  1. Stage 1 파인튜닝 제거 (v3 frozen 모델 유지)")
print("  2. Stage 2 전용 독립 backbone (공유 객체 버그 해결)")
print("  3. dropout 0.5 → 0.2 (BERT 파인튜닝 권장값)")
print("  4. v3 classifier 가중치 warm-start")
print("  5. lr_head 3e-4 → 1e-4 (warm-start 안정성)")
print("  6. Focal Loss (γ=2.0) 항상 적용")

print("\n" + "="*70)